# 08b | Logistic regression: modeling with scikit-learn

This is your first time using **scikit-learn** (sklearn for short), the standard machine learning library in Python. Every model in it works the same way, so learn one and you know them all:

```python
model = SomeModel(hyperparameters)   # 1. create
model.fit(X_train, y_train)          # 2. train
model.predict(X_test)                # 3. predict classes
model.predict_proba(X_test)          # 3'. predict probabilities (classifiers)
```

About 55 minutes.

## 0. First, hand-write two basic functions

"Write sigmoid by hand" and "write log loss by hand" are common interview requests.

### ✏️ Exercise 1 (Write it yourself)

Write a function `sigmoid(z)` whose input can be a number or a NumPy array, returning $\dfrac{1}{1 + e^{-z}}$. Use `np.exp`.

In [ ]:
import numpy as np

def sigmoid(z):
    # Write your code below
    pass

<details>
<summary><b>Answer</b></summary>

```python
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))
```

</details>

### ✏️ Exercise 2 (Write it yourself)

Write a function `log_loss(y, p)`: `y` is an array of 0/1 values and `p` the predicted probabilities of 1. Return the average log loss

$$-\frac{1}{n}\sum_i \big[y_i \log p_i + (1 - y_i)\log(1 - p_i)\big]$$

In [ ]:
def log_loss(y, p):
    # Write your code below
    pass

<details>
<summary><b>Answer</b></summary>

```python
def log_loss(y, p):
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
```

Predicting 0.5 for everything gives a log loss of ln 2 ≈ 0.693, the "knows nothing" baseline. Real implementations also clip p to (ε, 1−ε) to avoid log(0).

</details>

## 1. The full workflow: split → Pipeline → train → evaluate

Below we generate an imbalanced dataset (about 10% positives).

- `train_test_split`: splits into training and test sets. `stratify=y` keeps the share of positives the same in both; `random_state` is the random seed.
- `Pipeline`: chains "standardize" and "model" into a single object. When you call `fit` on it, the standardization parameters are computed **from the training data only**; when you call `predict`, new data is transformed with those same parameters automatically. This is the standard way to prevent data leakage.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_score, recall_score, average_precision_score

X, y = make_classification(n_samples=2000, n_features=10, n_informative=5,
                           weights=[0.9, 0.1], random_state=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=1)
print(X_train.shape, X_test.shape, y_train.mean().round(3), y_test.mean().round(3))

pipe = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000)),
])
pipe.fit(X_train, y_train)

`predict_proba` returns two columns: column 0 is the probability of class 0, column 1 the probability of class 1. We want **column 1**: `[:, 1]`.

Metric functions always take arguments in the order `(true values, predictions)`.

### ✏️ Exercise 3 (Fill in the blanks)

Use the trained `pipe` to get predicted probabilities `proba` on the test set, and compute the ROC-AUC `auc` and the PR-AUC `ap`.

In [ ]:
proba = pipe.predict_proba(____)[:, ____]
auc = roc_auc_score(y_test, ____)
ap = average_precision_score(y_test, ____)
print(round(auc, 4), round(ap, 4))

<details>
<summary><b>Answer</b></summary>

```python
proba = pipe.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, proba)
ap = average_precision_score(y_test, proba)
print(round(auc, 4), round(ap, 4))
```

</details>

## 2. Thresholds

`pipe.predict(X)` uses the default threshold of 0.5. To use a different threshold, compare the probabilities yourself: `(proba >= 0.2).astype(int)`.

`.astype(int)` turns True/False into 1/0. The precision and recall functions also take `(true values, 0/1 predictions)`:

In [ ]:
pred_30 = (proba >= 0.3).astype(int)
print(pred_30[:10])
print(precision_score(y_test, pred_30), recall_score(y_test, pred_30))

### ✏️ Exercise 4 (Write it yourself)

Turn `proba` into 0/1 predictions with thresholds 0.5 and 0.2, and compute four numbers: `prec_50`, `rec_50`, `prec_20`, `rec_20` (precision and recall).

In [ ]:
# Write your code below

<details>
<summary><b>Answer</b></summary>

```python
pred_50 = (proba >= 0.5).astype(int)
pred_20 = (proba >= 0.2).astype(int)
prec_50, rec_50 = precision_score(y_test, pred_50), recall_score(y_test, pred_50)
prec_20, rec_20 = precision_score(y_test, pred_20), recall_score(y_test, pred_20)
print(round(prec_50, 3), round(rec_50, 3))
print(round(prec_20, 3), round(rec_20, 3))
```

Lowering the threshold from 0.5 to 0.2: recall goes up, precision goes down. The model is unchanged; only the decision rule changed.

</details>

## 3. Regularization: Lasso and Ridge

sklearn's `LogisticRegression` **uses L2 regularization by default**. The strength is controlled by the parameter `C`, and note that it runs the other way: **`C` is the inverse of λ, so smaller `C` means stronger regularization.**

The data below have 30 features, of which only 5 are actually useful. Compare how many coefficients L1 and L2 push to 0.

- L1: `LogisticRegression(penalty="l1", solver="liblinear", C=0.05)`;
- The coefficients are in `model.coef_` (shape 1 × number of features). To get the model out of a Pipeline: `pipe.named_steps["model"]`.

In [ ]:
Xs, ys = make_classification(n_samples=1000, n_features=30, n_informative=5, n_redundant=0, random_state=3)

def fit_logit(penalty):
    p = Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(penalty=penalty, solver="liblinear", C=0.05)),
    ])
    p.fit(Xs, ys)
    return p.named_steps["model"].coef_[0]

coef_l2 = fit_logit("l2")
print(coef_l2.round(2))

### ✏️ Exercise 5 (Write it yourself)

Use `fit_logit` above to get the L1 coefficients `coef_l1`, then count how many coefficients are **exactly 0** under each type of regularization; store the counts in `n_zero_l1` and `n_zero_l2`.

In [ ]:
# Write your code below

<details>
<summary><b>Answer</b></summary>

```python
coef_l1 = fit_logit("l1")
n_zero_l1 = (coef_l1 == 0).sum()
n_zero_l2 = (coef_l2 == 0).sum()
print(n_zero_l1, n_zero_l2)
```

L1 pushes the coefficients of most useless features to 0; L2 pushes none. This is what "L1 produces sparse solutions" looks like in practice.

</details>

## 4. When you need p-values and odds ratios: use statsmodels

sklearn does not give standard errors or p-values. For inference use `smf.logit`, the equivalent of R's `glm(..., family = binomial)`. Exponentiating the coefficients gives odds ratios.

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf

rng = np.random.default_rng(0)
n = 2000
d = pd.DataFrame({"age": rng.normal(35, 10, n), "is_member": rng.integers(0, 2, n)})
logit_p = -3 + 0.04 * d["age"] + 0.7 * d["is_member"]
d["buy"] = (rng.random(n) < 1 / (1 + np.exp(-logit_p))).astype(int)

m = smf.logit("buy ~ age + is_member", data=d).fit(disp=0)
print(m.summary().tables[1])
print("odds ratios:")
print(np.exp(m.params).round(3))

**🗣 Say it out loud:** The odds ratio for `is_member` above is about 2. Explain it in one sentence for a business stakeholder.

<details>
<summary><b>Reference answer</b></summary>

"At the same age, members have about twice the odds of purchasing as non-members." If the listener isn't familiar with odds, switch to probabilities with a concrete case: "for a 35-year-old user, the purchase probability is about 17% for non-members and about 29% for members." Also point out that this is an association in observational data: members may simply be people who buy more anyway, so you can't say "becoming a member doubles the odds of purchasing".

**Short version:** Holding age fixed, members have about twice the odds of purchasing. I'd translate that into probabilities for a typical user, and caveat that it's an association, since members may self-select.

</details>

## Summary

- sklearn in three steps: create → `fit` → `predict` / `predict_proba`;
- Put preprocessing and the model in a `Pipeline`; split with `stratify=y`;
- Take probabilities with `[:, 1]` and choose the threshold yourself;
- Smaller `C` means stronger regularization; L1 produces sparse coefficients;
- For p-values and odds ratios use `smf.logit`.

On to **09a**.